Created a seperate file to scrape the data from the f1 offical website to be used in Slide 6 of my analysis. The website used is located here: https://www.formula1.com/en/results/1993/races
- To find the necessary data, change year and circuit. I am only scraping qualifying data for pre-1994 seasons

## Monaco qualifying data pre-1994

In [1]:
# IDEA: make list of years then loop over this list of years to add this as a column to the dataframe which corresponds to the race year scraped from)
# 1950 monaco grand prix qualifying results page: https://www.formula1.com/en/results/1950/races/95/monaco/qualifying/0
# 1993 monaco grand prix qualifying results page: https://www.formula1.com/en/results/1993/races/595/monaco/qualifying/0
# 1955 monaco grand prix qualifying results page: https://www.formula1.com/en/results/1955/races/136/monaco/qualifying/0

import re # need regex for parsing html
import time # to prevent 403 errors from too many requests
import requests
import pandas as pd
from io import StringIO # to prevent warings from pd.read_html about deprecated behavior

In [2]:

# make gloabl variables to be used in functions
BASE = "https://www.formula1.com"
UA = {"User-Agent": "Mozilla/5.0"}

# get the race ids from the races page for each year. Returns the race id
def get_monaco_race_id(year: int) -> int | None: # expects an int, returns an int or None
    url = f"{BASE}/en/results/{year}/races"
    html = requests.get(url, headers=UA, timeout=30).text

    # Match any Monaco URL on that page, regardless of session
    # e.g. /en/results/1950/races/95/monaco/race-result
    m = re.search(rf"/en/results/{year}/races/(\d+)/monaco/", html) # regex (\d+) captures the race id
    return int(m.group(1)) if m else None # group(1) is the first capture group from the regex

# fetch the qualifying results for Monaco for a given year
def fetch_monaco_qualifying(year: int) -> pd.DataFrame | None:
    race_id = get_monaco_race_id(year)
    if race_id is None:
        return None

    qual_url = f"{BASE}/en/results/{year}/races/{race_id}/monaco/qualifying/0"
    html = requests.get(qual_url, headers=UA, timeout=30).text

    df = pd.read_html(StringIO(html))[0].copy() # wrap in StringIO to avoid warnings
    df["Year"] = year
    df["RaceId"] = race_id
    df["SourceURL"] = qual_url
    return df

dfs, skipped = [], []

for year in range(1950, 2025):
    df = fetch_monaco_qualifying(year)
    if df is None:
        skipped.append(year)
    else:
        dfs.append(df)
    time.sleep(0.5)

# stack all DataFrames in dfs vertically (row-wise) into one big DataFrame.
# need else clause to handle case where no data was fetched
all_quali_monaco = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f"skipped: {skipped}")

skipped: [1951, 1952, 1953, 1954, 2020]


In [3]:
# rename columns
all_quali_monaco.rename(
    columns={"Driver": "driverName", "Time": "best_quali_time"},
    inplace=True
)

# function to remove final three letters
def remove_final_three_letters(name: str) -> str:
    return name[:-3] if len(name) > 3 else name

# drop column and update driverName
all_quali_monaco = all_quali_monaco.drop(columns=['SourceURL'])
all_quali_monaco['driverName'] = all_quali_monaco['driverName'].apply(remove_final_three_letters)


In [4]:
all_quali_monaco.to_csv("monaco_grand_prix_qualifying_1950_2025.csv", index=False)

## Monza (Italian grand prix) qualifying data pre-1994

In [5]:
# Website EXAMPLES:
# https://www.formula1.com/en/results/1993/races/601/italy/qualifying/0
# https://www.formula1.com/en/results/1976/races/370/italy/qualifying/0


In [6]:
# make gloabl variables to be used in functions
BASE = "https://www.formula1.com"
UA = {"User-Agent": "Mozilla/5.0"}

# get the race ids from the races page for each year. Returns the race id
def get_italy_race_id(year: int) -> int | None: # expects an int, returns an int or None
    url = f"{BASE}/en/results/{year}/races"
    html = requests.get(url, headers=UA, timeout=30).text

    # Match any Monaco URL on that page, regardless of session
    # e.g. /en/results/1950/races/95/monaco/race-result
    m = re.search(rf"/en/results/{year}/races/(\d+)/italy/", html) # regex (\d+) captures the race id
    return int(m.group(1)) if m else None # group(1) is the first capture group from the regex

# fetch the qualifying results for Monaco for a given year
def fetch_italy_qualifying(year: int) -> pd.DataFrame | None:
    race_id = get_italy_race_id(year)
    if race_id is None:
        return None

    qual_url = f"{BASE}/en/results/{year}/races/{race_id}/italy/qualifying/0"
    html = requests.get(qual_url, headers=UA, timeout=30).text

    df = pd.read_html(StringIO(html))[0].copy() # wrap in StringIO to avoid warnings
    df["Year"] = year
    df["RaceId"] = race_id
    df["SourceURL"] = qual_url
    return df

dfs, skipped = [], []

for year in range(1950, 2025):
    df = fetch_italy_qualifying(year)
    if df is None:
        skipped.append(year)
    else:
        dfs.append(df)
    time.sleep(0.5)

# stack all DataFrames in dfs vertically (row-wise) into one big DataFrame.
# need else clause to handle case where no data was fetched
all_quali_italy = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f"skipped: {skipped}")

skipped: []


In [7]:
# rename columns
all_quali_italy.rename(
    columns={"Driver": "driverName", "Time": "best_quali_time"},
    inplace=True
)

# function to remove final three letters
def remove_final_three_letters(name: str) -> str:
    return name[:-3] if len(name) > 3 else name

# drop column and update driverName
all_quali_italy = all_quali_italy.drop(columns=['SourceURL'])
all_quali_italy['driverName'] = all_quali_italy['driverName'].apply(remove_final_three_letters)

In [8]:
all_quali_italy.to_csv("italian_grand_prix_qualifying_1950_2025.csv", index=False)